In [1]:
from google.colab import drive
drive.mount('/content/drive')

dir_ = '/content/drive/MyDrive/Cohand/Minh học data/Kỳ 3/NLP/'
raw_data = dir_ + 'data/raw/'
processed_data = dir_ + 'data/processed/'
requirment_file = dir_ + 'requirements.txt'
output = dir_ + 'output/'

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [3]:
def save_report_to_csv(y_true, y_pred, target_names, model_name, filename=None):
    """
    Lưu classification report ra file CSV.
    """
    report_dict = classification_report(y_true, y_pred, target_names=target_names, zero_division=0, output_dict=True)

    df_report = pd.DataFrame(report_dict).transpose()

    df_report['Model'] = model_name
    cols = ['Model'] + [col for col in df_report.columns if col != 'Model']
    df_report = df_report[cols]

    if filename is None:
        clean_model_name = model_name.replace(' ', '_').replace('+', '').lower()
        filename = f"report_{clean_model_name}.csv"

    df_report.to_csv(output + filename, index=True, index_label='Label')
    print(f"Saved the report of [{model_name}] into: {filename}")

    return df_report

In [4]:
df = pd.read_csv(processed_data+'processed.csv')
df = df[df['processed_data'].notna()]
print(df.shape)
df.head(5)

(15778, 11)


,data,stayingpower,texture,smell,price,others,colour,shipping,packing,processed_data,labels
0,Công dụng: tốt\r\nKết cấu: đẹp\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng tốt kết_cấu đẹp độ bền màu lâu,"['stayingpower_positive', 'texture_positive']"
1,Công dụng: son môi\r\nKết cấu: khô\r\nĐộ bền m...,positive,positive,NaN,NaN,NaN,NaN,NaN,NaN,công_dụng son môi kết_cấu khô độ bền màu tuyệt...,"['stayingpower_positive', 'texture_positive']"
2,"Son mịn, mùi thơm nhẹ, lâu trôi.\r\nVideo+ hìn...",positive,positive,positive,NaN,NaN,NaN,NaN,NaN,son mịn mùi thơm nhẹ lâu trôi video hình_ảnh m...,"['stayingpower_positive', 'texture_positive', ..."
3,Công dụng: đánh son\r\nKết cấu: Đóng gói cẩn t...,positive,NaN,NaN,positive,NaN,NaN,negative,NaN,công_dụng đánh son kết_cấu đóng_gói cẩn_thận đ...,"['stayingpower_positive', 'price_positive', 's..."
4,Công dụng: tốt\r\nKết cấu: tốt\r\nĐộ bền màu: ...,positive,positive,NaN,NaN,NaN,neutral,NaN,NaN,công_dụng tốt kết_cấu tốt độ bền màu tốt đóng_...,"['stayingpower_positive', 'texture_positive', ..."


In [5]:
aspects = ['stayingpower', 'texture', 'smell', 'price', 'others', 'colour', 'shipping', 'packing']

def extract_labels(row):
    labels = []
    for aspect in aspects:
        if pd.notna(row[aspect]):
            labels.append(f"{aspect}_{row[aspect].strip().lower()}")
    return labels

df['labels'] = df.apply(extract_labels, axis=1)

# Chuyển danh sách nhãn thành ma trận nhị phân
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['labels'])

X = df['processed_data'].values

print(f"Num of rows after cleaning: {len(X)}")
print("List of the label classes:", mlb.classes_)

# Split Train / Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set shape: {X_train.shape[0]}, Test set shape: {X_test.shape[0]}")

Num of rows after cleaning: 15778
List of the label classes: ['colour_negative' 'colour_neutral' 'colour_positive' 'others_neutral'
 'packing_negative' 'packing_neutral' 'packing_positive' 'price_negative'
 'price_neutral' 'price_positive' 'shipping_negative' 'shipping_neutral'
 'shipping_positive' 'smell_negative' 'smell_neutral' 'smell_positive'
 'stayingpower_negative' 'stayingpower_neutral' 'stayingpower_positive'
 'texture_negative' 'texture_neutral' 'texture_positive']
Train set shape: 12622, Test set shape: 3156


# PhoBERT

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.metrics import classification_report

# 1. Định nghĩa Dataset class cho PyTorch
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")
        item = {key: val.squeeze() for key, val in inputs.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# 2. Load Tokenizer và Model
model_name = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=y.shape[1],
    problem_type="multi_label_classification"
)

# 3. Tạo Dataset
train_dataset = ReviewDataset(X_train, y_train, tokenizer)
test_dataset = ReviewDataset(X_test, y_test, tokenizer)

# 4. Tạo funtion tính F1_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()

    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, predictions, average='micro', zero_division=0)

    return {'macro_f1': macro_f1, 'micro_f1': micro_f1}

# 5. Training setup
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# 6. Training
trainer.train()

trainer.evaluate()

print("Predicting on PhoBERT...")
predictions_output = trainer.predict(test_dataset)

# Lấy giá trị logits
logits = predictions_output.predictions

# Áp dụng Sigmoid để chuyển logits thành xác suất (từ 0 đến 1)
probs = torch.sigmoid(torch.tensor(logits)).numpy()

# Chuyển thành xác suất thành nhãn nhị phân dựa trên threshold
threshold = 0.5
y_pred_phobert = (probs > threshold).astype(int)

y_true_phobert = predictions_output.label_ids

# Save report
print(classification_report(y_true_phobert, y_pred_phobert, target_names=mlb.classes_, zero_division=0))
df_phobert = save_report_to_csv(y_true_phobert, y_pred_phobert, mlb.classes_, "PhoBERT")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Predicting on PhoBERT...
                       precision    recall  f1-score   support

      colour_negative       0.71      0.13      0.22       134
       colour_neutral       0.00      0.00      0.00       105
      colour_positive       0.95      0.95      0.95      1350
       others_neutral       0.96      0.93      0.94       480
     packing_negative       0.00      0.00      0.00        20
      packing_neutral       0.00      0.00      0.00         2
     packing_positive       0.94      0.98      0.96       600
       price_negative       0.00      0.00      0.00         7
        price_neutral       0.00      0.00      0.00         8
       price_positive       0.95      0.98      0.97       663
    shipping_negative       0.88      0.94      0.91       326
     shipping_neutral       0.00      0.00      0.00        74
    shipping_positive       0.93      0.96      0.95       666
       smell_negative       1.00      0.01      0.02        97
        smell_neutral       0

In [7]:
y_pred_phobert

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1]])

# XLM-RoBERTa

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.metrics import classification_report

# 1. Định nghĩa Dataset class cho PyTorch
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors="pt")
        item = {key: val.squeeze() for key, val in inputs.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

# 2. Load Tokenizer và Model
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=y.shape[1],
    problem_type="multi_label_classification"
)

# 3. Tạo Dataset
train_dataset = ReviewDataset(X_train, y_train, tokenizer)
test_dataset = ReviewDataset(X_test, y_test, tokenizer)

# 4. Tạo funtion tính F1_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()

    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    micro_f1 = f1_score(labels, predictions, average='micro', zero_division=0)

    return {'macro_f1': macro_f1, 'micro_f1': micro_f1}

# 5. Training setup
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# 6. Train
trainer.train()

trainer.evaluate()

print("Predicting RoBERTa...")
predictions_output = trainer.predict(test_dataset)

# Lấy giá trị logits
logits = predictions_output.predictions

# Áp dụng Sigmoid để chuyển logits thành xác suất (từ 0 đến 1)
probs = torch.sigmoid(torch.tensor(logits)).numpy()

# Chuyển thành xác suất thành nhãn nhị phân dựa trên threshold
threshold = 0.5
y_pred_phobert = (probs > threshold).astype(int)

y_true_phobert = predictions_output.label_ids

# Save report
print(classification_report(y_true_phobert, y_pred_phobert, target_names=mlb.classes_, zero_division=0))

df_phobert = save_report_to_csv(y_true_phobert, y_pred_phobert, mlb.classes_, "XLM_RoBERTa")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Predicting RoBERTa...
                       precision    recall  f1-score   support

      colour_negative       0.71      0.50      0.59       134
       colour_neutral       0.00      0.00      0.00       105
      colour_positive       0.94      0.95      0.95      1350
       others_neutral       0.96      0.91      0.94       480
     packing_negative       0.00      0.00      0.00        20
      packing_neutral       0.00      0.00      0.00         2
     packing_positive       0.92      0.98      0.95       600
       price_negative       0.00      0.00      0.00         7
        price_neutral       0.00      0.00      0.00         8
       price_positive       0.95      0.98      0.96       663
    shipping_negative       0.87      0.92      0.90       326
     shipping_neutral       0.00      0.00      0.00        74
    shipping_positive       0.94      0.96      0.95       666
       smell_negative       0.87      0.69      0.77        97
        smell_neutral       0.00